![DB Academy](./Includes/images/db-academy.png)

<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif;">
<div style="background: #02A36F; color: white; border-radius: 8px; padding: 28px 32px; text-align: center; position: relative;">
  <div style="font-size: 14pt; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85; margin-bottom: 8px;">Lab</div>
  <div style="font-size: 24pt; font-weight: 700; line-height: 1.3;">Incrementally Load New Orders</div>
  <div style="font-size: 14pt; margin-top: 12px; opacity: 0.9;">Use Auto Loader to add Q1 and Q2 2025 data to the orders_bronze table with AI helping you write it.</div>
</div>
</div>


## REQUIRED: SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

This notebook runs on Serverless compute. Before running any cells, confirm Serverless is selected in the compute dropdown at the top-right of this notebook. This course was developed and tested on <strong>Serverless environment version 5</strong>.
  </div>
</div>

### Setup
Run the cell below to configure your environment for this lesson. The setup script will initialize your catalog and schema so your queries run against the correct context.


In [0]:
%run ./Includes/Classroom-Setup-07

Workspace,User Catalog Name,Catalog Status
Learner Workspace,learn_get_started_dw_joemar_brillo,Catalog Already Exists


DataFrame[]

DataFrame[]

DataFrame[]

**Context.** In Lesson 06 you used Auto Loader to load the four 2024 quarters into `orders_bronze`. The data team has now landed two more files in your volume: `orders_2025_q1.csv` and `orders_2025_q2.csv`. Your job is to ingest them into the same table and confirm everything looks right.

In this practice you will:

1. Confirm the starting state of `orders_bronze`.
2. Use Auto Loader to ingest the two 2025 files.
3. Re-run the load to confirm exactly-once safety.
4. Choose what to verify, and write that query.

**How to work each task:** the default path is **Genie Code**. Open it inline with **Cmd/Ctrl + I** in the target cell, or use the full **Genie / Assistant** panel, and give it the prompt provided. Run what it generates, then expand the reference to compare. Letting AI write the Auto Loader call is the point here, you focus on *what* you want ingested, not the boilerplate.


### Task 1: Confirm the starting state

Before loading anything new, confirm `orders_bronze` already holds the 202,984 rows from the four 2024 quarters.

**Genie Code prompt:**

<div style="position:relative; margin:6px 0;">
<button onclick="var t=document.getElementById('gp_557_1').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:5px 11px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#EEF3F5; border-left:3px solid #1B5162; border-radius:6px; padding:12px 14px; margin:0; font-size:14px; white-space:pre-wrap;"><code id="gp_557_1">Count the rows in the orders_bronze table.</code></pre>
</div>


In [0]:
%sql
-- Use Genie Code with the prompt above, then run it.
SELECT COUNT(*) FROM orders_bronze;

COUNT(*)
202984


<details>
<summary><strong>Here is the count query</strong></summary>
<div style="position:relative; margin-top:8px;">
<button onclick="var t=document.getElementById('cb12').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:6px 12px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#F9F7F4; border-radius:6px; padding:14px; margin:0; font-size:14px; overflow:auto;"><code id="cb12">SELECT COUNT(*) AS row_count FROM orders_bronze;</code></pre>
</div>
</details>

You should see `202984`. If you see `0` or a "table not found" error, the setup backfill will have rebuilt it, re-run the setup cell at the top and try again.


### Task 2: Ingest the 2025 files with Auto Loader

Use Auto Loader to ingest **both** 2025 files into `orders_bronze` in one go. Auto Loader's checkpoint lives in your data volume at `data/_ckpt_orders_bronze` (the demo and setup use the same one), so pointing at it means the four 2024 files are already known and only the 2025 files get ingested.

Rather than hand-writing the streaming code, describe it to **Genie Code** and let it build the call. Open Genie Code in the cell below and use this prompt:

> Use Auto Loader (the cloudFiles format) to read CSV files matching the glob orders_2025_q*.csv from the data volume for the current catalog, with header true and inferred column types, using schema location data/_schema_orders_bronze, and write a one-time availableNow batch into the orders_bronze table using checkpoint location data/_ckpt_orders_bronze.

Run what Genie generates, then compare with the reference.


In [0]:
# # Auto Loader: ingest the two 2025 quarterly files into orders_bronze (one-time availableNow batch)
# catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
# base = f"/Volumes/{catalog}/vintage_audio/data"

# (spark.readStream
#     .format("cloudFiles")
#     .option("cloudFiles.format", "csv")
#     .option("cloudFiles.schemaLocation", f"{base}/_schema_orders_bronze")
#     .option("header", "true")
#     .option("cloudFiles.inferColumnTypes", "true")
#     .option("pathGlobFilter", "orders_2025_q*.csv")
#     .load(base)
#   .writeStream
#     .option("checkpointLocation", f"{base}/_ckpt_orders_bronze")
#     .trigger(availableNow=True)
#     .toTable(f"{catalog}.vintage_audio.orders_bronze")
#     .awaitTermination())
catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
base = f"/Volumes/{catalog}/vintage_audio/data"
(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", f"{base}/_schema_orders_bronze")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("pathGlobFilter", "orders_2025_q*.csv")
    .load(base)
  .writeStream
    .option("checkpointLocation", f"{base}/_ckpt_orders_bronze")
    .trigger(availableNow=True)
    .toTable(f"{catalog}.vintage_audio.orders_bronze")
    .awaitTermination())

<details>
<summary><strong>Here is the full Auto Loader load</strong></summary>
<div style="position:relative; margin-top:8px;">
<button onclick="var t=document.getElementById('cb13').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:6px 12px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#F9F7F4; border-radius:6px; padding:14px; margin:0; font-size:14px; overflow:auto;"><code id="cb13">catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
base = f"/Volumes/{catalog}/vintage_audio/data"
(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", f"{base}/_schema_orders_bronze")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("pathGlobFilter", "orders_2025_q*.csv")
    .load(base)
  .writeStream
    .option("checkpointLocation", f"{base}/_ckpt_orders_bronze")
    .trigger(availableNow=True)
    .toTable(f"{catalog}.vintage_audio.orders_bronze")
    .awaitTermination())</code></pre>
</div>
<div style="margin-top:8px; font-size:14px;">This is the same pattern the lesson 06 load used. Genie may format it slightly differently, what matters is the cloudFiles source, the glob, the shared checkpoint, and the availableNow trigger.</div>
</details>

Auto Loader ingests `orders_2025_q1.csv` (51,826 rows) and `orders_2025_q2.csv` (43,188 rows). Confirm the new total:

**Genie Code prompt:**

<div style="position:relative; margin:6px 0;">
<button onclick="var t=document.getElementById('gp_557_2').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:5px 11px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#EEF3F5; border-left:3px solid #1B5162; border-radius:6px; padding:12px 14px; margin:0; font-size:14px; white-space:pre-wrap;"><code id="gp_557_2">Count the rows in orders_bronze again.</code></pre>
</div>


In [0]:
%sql
-- Use Genie Code with the prompt above, then run it.
SELECT COUNT(*) AS row_count FROM orders_bronze;

row_count
202984


<details>
<summary><strong>Here is the count query</strong></summary>
<div style="position:relative; margin-top:8px;">
<button onclick="var t=document.getElementById('cb14').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:6px 12px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#F9F7F4; border-radius:6px; padding:14px; margin:0; font-size:14px; overflow:auto;"><code id="cb14">SELECT COUNT(*) AS row_count FROM orders_bronze;</code></pre>
</div>
</details>

You should now see **297,998** (202,984 from 2024 + 51,826 + 43,188 from 2025).


### Task 3: Re-run to confirm exactly-once

Run the **exact same** Auto Loader load from Task 2 again. Because the checkpoint already lists both 2025 files, the stream should ingest **zero** new rows.

Re-run your Task 2 cell (or copy the reference again), then check the count once more. It should still be **297,998**, the re-run changed nothing. That's exactly-once ingestion: safe to run on any schedule.


In [0]:
%sql
-- After re-running the Auto Loader load, confirm the count is unchanged.
SELECT COUNT(*) AS row_count FROM orders_bronze;

row_count
202984


### Task 4: Choose what to verify

A good engineer doesn't just check the row count, they sanity-check the data itself. **Pick one (or more)** of the checks below, then use **Genie Code** to write the query. Each is a real thing you'd verify after a production load.

| Check | Genie Code prompt to try |
|---|---|
| **Date coverage** | "Show the earliest and latest order_date in orders_bronze." |
| **No duplicate order IDs** | "Count how many order_id values appear more than once in orders_bronze." |
| **Revenue by year** | "Sum total_amount by the year of order_date in orders_bronze." |

Write your chosen check in the cell below. A reference for the date-coverage check is provided.


In [0]:
%sql
-- Use Genie Code with the prompt above, then run it.
SELECT
  MIN(order_date) AS earliest_order,
  MAX(order_date) AS latest_order
FROM orders_bronze;

earliest_order,latest_order
2024-01-01,2024-12-31


In [0]:
# Task 4: No duplicate order IDs check
# Count how many order_id values appear more than once in orders_bronze.
display(spark.sql("""
SELECT COUNT(*) AS duplicate_order_ids
FROM (
  SELECT order_id
  FROM orders_bronze
  GROUP BY order_id
  HAVING COUNT(*) > 1
)
"""))

duplicate_order_ids
0


In [0]:
# Task 4: Revenue by year check
# Sum total_amount by the year of order_date in orders_bronze.
display(spark.sql("""
SELECT
  YEAR(order_date) AS order_year,
  SUM(total_amount) AS total_revenue
FROM orders_bronze
GROUP BY YEAR(order_date)
ORDER BY order_year
"""))

<details>
<summary><strong>Here is the date-coverage query</strong></summary>
<div style="position:relative; margin-top:8px;">
<button onclick="var t=document.getElementById('cb15').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:6px 12px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#F9F7F4; border-radius:6px; padding:14px; margin:0; font-size:14px; overflow:auto;"><code id="cb15">SELECT
  MIN(order_date) AS earliest_order,
  MAX(order_date) AS latest_order
FROM orders_bronze;</code></pre>
</div>
</details>

With both 2025 files loaded, the date coverage should extend through mid-2025. The duplicate-ID check should return zero, and revenue-by-year should show both 2024 and 2025. Whichever check you chose, it confirms the new data landed cleanly on top of the 2024 data - **297,998** rows in total.


## Nice work
<div style="border-left: 4px solid #4caf50; background: #e8f5e9; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
<div style="color:#333;">
In this lab you:

- Confirmed the starting state of <code>orders_bronze</code>.
- Used <strong>Genie Code</strong> to build an <strong>Auto Loader</strong> load for the two 2025 files.
- Re-ran the load and confirmed <strong>exactly-once</strong> ingestion (no duplicates).
- Chose your own verification check and wrote it with AI.

</div>
</div>


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>